# C9-dimensionality-reduction — Practice p23

**Type:** integrative · **Difficulty:** advanced · **Concepts:** pca-centered-covariance-eigenproblem-derivation, numpy-pca-class-from-scratch, pca-black-box-insufficiency

Extend `NumpyPCA` into the complete reusable API from Session 2.
The `fit` and `transform` contracts are exactly p22's: sample covariance divides by `n - 1`, one `np.linalg.eigh` call per successful fit, descending paired eigenvalues/components, full-spectrum explained ratios, validation before decomposition, no input mutation, and learned-state reuse.

Add:

- `fit_transform(X)`, which performs one fit and returns `transform(X)`;
- `inverse_transform(Z)`, which rejects use before fit and invalid/non-finite/wrong-width score matrices, then returns `Z @ components_ + mean_`.

The checker integrates all three targets.
It verifies the covariance/SVD spectrum and retained projector, proves reconstruction equals centered projection, checks the dropped spectral-tail error, and requires exact full-rank reconstruction within `ATOL = 1e-10`, `RTOL = 0.0`.

**Zero points:** sklearn PCA, scipy PCA, or helpers that invoke them.
Do not hard-code the public fixtures.


In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0


class NumpyPCA:
    def __init__(self, n_components):
        # YOUR CODE HERE
        ...

    def fit(self, X):
        # YOUR CODE HERE
        ...

    def transform(self, X):
        # YOUR CODE HERE
        ...

    def fit_transform(self, X):
        # YOUR CODE HERE
        ...

    def inverse_transform(self, Z):
        # YOUR CODE HERE
        ...

## Immutable contract check — do not edit

Every comparison has `rtol=0.0`.
Direction signs are removed through projectors, and reconstruction is checked independently through covariance and centered SVD.


In [ ]:
import dis
import inspect
import types

_ORIGINAL_EIGH_P23 = np.linalg.eigh


def _audit_numpy_pca_p23():
    pending = []
    for value in vars(NumpyPCA).values():
        if isinstance(value, (staticmethod, classmethod)):
            value = value.__func__
        if isinstance(value, types.FunctionType):
            pending.append(value)
    seen = set()
    while pending:
        function = pending.pop()
        if id(function) in seen:
            continue
        seen.add(id(function))
        codes = [function.__code__]
        while codes:
            code = codes.pop()
            codes.extend(item for item in code.co_consts if isinstance(item, types.CodeType))
            assert not ({"sklearn", "scipy"} & {name.lower() for name in code.co_names})
            for instruction in dis.get_instructions(code):
                if instruction.opname in {"IMPORT_NAME", "IMPORT_FROM"}:
                    assert not str(instruction.argval).lower().startswith(("sklearn", "scipy"))
            for name in code.co_names:
                value = function.__globals__.get(name)
                if isinstance(value, types.FunctionType):
                    pending.append(value)
                assert not str(getattr(value, "__module__", "")).lower().startswith(("sklearn", "scipy"))
    try:
        source = inspect.getsource(NumpyPCA).lower()
    except (OSError, TypeError):
        source = ""
    assert "sklearn" not in source and "scipy" not in source


_audit_numpy_pca_p23()

_X_p23 = np.array([
    [5.0, 1.0, 0.0, 2.0],
    [2.0, 4.0, 1.0, 0.0],
    [0.0, 2.0, 5.0, 1.0],
    [3.0, -1.0, 2.0, 4.0],
    [6.0, 3.0, -2.0, 1.0],
    [1.0, 0.0, 3.0, 5.0],
])
_X_before_p23 = _X_p23.copy()
_eigh_calls_p23 = []
def _traced_eigh_p23(matrix):
    _eigh_calls_p23.append(np.array(matrix, copy=True))
    return _ORIGINAL_EIGH_P23(matrix)
np.linalg.eigh = _traced_eigh_p23
try:
    _model_p23 = NumpyPCA(2)
    _fit_calls_p23 = []
    _original_fit_p23 = _model_p23.fit
    def _traced_fit_p23(X):
        _fit_calls_p23.append(np.array(X, copy=True))
        return _original_fit_p23(X)
    _model_p23.fit = _traced_fit_p23
    _Z_p23 = _model_p23.fit_transform(_X_p23)
finally:
    np.linalg.eigh = _ORIGINAL_EIGH_P23
assert len(_fit_calls_p23) == 1 and len(_eigh_calls_p23) == 1
assert np.array_equal(_X_p23, _X_before_p23)
assert isinstance(_Z_p23, np.ndarray) and _Z_p23.shape == (6, 2)
assert np.issubdtype(_Z_p23.dtype, np.floating) and np.isfinite(_Z_p23).all()

_mean_ref_p23 = _X_p23.mean(axis=0)
_Xc_ref_p23 = _X_p23 - _mean_ref_p23
_C_ref_p23 = _Xc_ref_p23.T @ _Xc_ref_p23 / (_X_p23.shape[0] - 1)
assert np.allclose(_eigh_calls_p23[0], _C_ref_p23, atol=ATOL, rtol=RTOL)
_evals_ref_p23, _evecs_ref_p23 = _ORIGINAL_EIGH_P23(_C_ref_p23)
_order_ref_p23 = np.argsort(_evals_ref_p23)[::-1]
_evals_ref_p23 = np.maximum(_evals_ref_p23[_order_ref_p23], 0.0)
_evecs_ref_p23 = _evecs_ref_p23[:, _order_ref_p23]
_, _s_ref_p23, _Vt_ref_p23 = np.linalg.svd(_Xc_ref_p23, full_matrices=False)
_P_ref_p23 = _Vt_ref_p23[:2].T @ _Vt_ref_p23[:2]
_P_model_p23 = _model_p23.components_.T @ _model_p23.components_
for _name_p23, _shape_p23 in (
    ("mean_", (4,)),
    ("components_", (2, 4)),
    ("explained_variance_", (2,)),
    ("explained_variance_ratio_", (2,)),
):
    _value_p23 = getattr(_model_p23, _name_p23)
    assert isinstance(_value_p23, np.ndarray) and _value_p23.shape == _shape_p23
    assert np.issubdtype(_value_p23.dtype, np.floating) and np.isfinite(_value_p23).all()
assert np.allclose(_model_p23.mean_, _mean_ref_p23, atol=ATOL, rtol=RTOL)
assert np.allclose(_model_p23.components_ @ _model_p23.components_.T, np.eye(2), atol=ATOL, rtol=RTOL)
assert np.allclose(_model_p23.explained_variance_, _s_ref_p23[:2]**2 / 5, atol=ATOL, rtol=RTOL)
assert np.allclose(_model_p23.explained_variance_, _evals_ref_p23[:2], atol=ATOL, rtol=RTOL)
assert np.allclose(_model_p23.explained_variance_ratio_, _evals_ref_p23[:2] / _evals_ref_p23.sum(), atol=ATOL, rtol=RTOL)
assert np.allclose(_P_model_p23, _P_ref_p23, atol=ATOL, rtol=RTOL)
assert np.allclose(_Z_p23, _Xc_ref_p23 @ _model_p23.components_.T, atol=ATOL, rtol=RTOL)

_Xhat_p23 = _model_p23.inverse_transform(_Z_p23)
_Xhat_ref_p23 = _Xc_ref_p23 @ _P_model_p23 + _mean_ref_p23
assert isinstance(_Xhat_p23, np.ndarray) and _Xhat_p23.shape == _X_p23.shape
assert np.allclose(_Xhat_p23, _Xhat_ref_p23, atol=ATOL, rtol=RTOL)
_error2_p23 = ((_X_p23 - _Xhat_p23) ** 2).sum()
_tail2_p23 = (_X_p23.shape[0] - 1) * _evals_ref_p23[2:].sum()
assert np.isclose(_error2_p23, _tail2_p23, atol=ATOL, rtol=RTOL)

np.linalg.eigh = _traced_eigh_p23
try:
    _full_p23 = NumpyPCA(_X_p23.shape[1])
    _full_Z_p23 = _full_p23.fit_transform(_X_p23)
    _full_hat_p23 = _full_p23.inverse_transform(_full_Z_p23)
finally:
    np.linalg.eigh = _ORIGINAL_EIGH_P23
assert len(_eigh_calls_p23) == 2
assert np.allclose(_full_hat_p23, _X_p23, atol=ATOL, rtol=RTOL)

# A controlled refit changes both mean and covariance. Every learned array and
# both directions of the reusable API must consume the returned eigenpairs.
_X_refit_p23 = np.array([
    [12.0, -4.0, 8.0, 1.0], [18.0, 3.0, 2.0, -5.0],
    [9.0, 7.0, -3.0, 4.0], [21.0, -2.0, 6.0, 9.0],
    [15.0, 5.0, 11.0, -1.0], [7.0, -6.0, 0.0, 12.0],
])
_mean_refit_p23 = _X_refit_p23.mean(axis=0)
_Xc_refit_p23 = _X_refit_p23 - _mean_refit_p23
_C_refit_p23 = _Xc_refit_p23.T @ _Xc_refit_p23 / (_X_refit_p23.shape[0] - 1)
assert not np.allclose(_C_refit_p23, _C_ref_p23, atol=ATOL, rtol=RTOL)
_sentinel_values_p23 = np.array([5.5, 17.0, 2.25, 9.0])
_sentinel_vectors_p23 = np.array([
    [0.0, 0.0, 1.0, 0.0],
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
    [0.0, 1.0, 0.0, 0.0],
])
_controlled_calls_p23 = []
def _controlled_eigh_p23(matrix):
    _controlled_calls_p23.append(np.array(matrix, copy=True))
    assert np.allclose(matrix, _C_refit_p23, atol=ATOL, rtol=RTOL)
    return _sentinel_values_p23.copy(), _sentinel_vectors_p23.copy()
_before_refit_p23 = _X_refit_p23.copy()
np.linalg.eigh = _controlled_eigh_p23
try:
    _returned_refit_p23 = _model_p23.fit(_X_refit_p23)
finally:
    np.linalg.eigh = _ORIGINAL_EIGH_P23
assert _returned_refit_p23 is _model_p23 and len(_controlled_calls_p23) == 1
assert np.array_equal(_X_refit_p23, _before_refit_p23)
_sentinel_order_p23 = np.argsort(_sentinel_values_p23)[::-1]
_expected_values_p23 = _sentinel_values_p23[_sentinel_order_p23]
_expected_components_p23 = _sentinel_vectors_p23[:, _sentinel_order_p23[:2]].T
assert np.array_equal(_model_p23.mean_, _mean_refit_p23)
assert np.array_equal(_model_p23.explained_variance_, _expected_values_p23[:2])
assert np.array_equal(
    _model_p23.explained_variance_ratio_,
    _expected_values_p23[:2] / _expected_values_p23.sum(),
)
for _j_controlled_p23 in range(2):
    _P_model_controlled_p23 = np.outer(
        _model_p23.components_[_j_controlled_p23],
        _model_p23.components_[_j_controlled_p23],
    )
    _P_expected_controlled_p23 = np.outer(
        _expected_components_p23[_j_controlled_p23],
        _expected_components_p23[_j_controlled_p23],
    )
    assert np.allclose(
        _P_model_controlled_p23, _P_expected_controlled_p23,
        atol=ATOL, rtol=RTOL,
    )
_probe_X_p23 = np.array([[25.0, -8.0, 14.0, 3.0], [4.0, 9.0, -5.0, 16.0]])
_probe_X_before_p23 = _probe_X_p23.copy()
_probe_Z_p23 = _model_p23.transform(_probe_X_p23)
_probe_Z_before_p23 = _probe_Z_p23.copy()
assert np.allclose(
    _probe_Z_p23, (_probe_X_p23 - _mean_refit_p23) @ _model_p23.components_.T,
    atol=ATOL, rtol=RTOL,
)
_probe_hat_p23 = _model_p23.inverse_transform(_probe_Z_p23)
assert np.allclose(
    _probe_hat_p23,
    _probe_Z_p23 @ _model_p23.components_ + _mean_refit_p23,
    atol=ATOL, rtol=RTOL,
)
assert np.array_equal(_probe_X_p23, _probe_X_before_p23)
assert np.array_equal(_probe_Z_p23, _probe_Z_before_p23)

_invalid_fit_p23 = (
    (np.ones(4), 1),
    (np.ones((1, 4)), 1),
    (np.ones((3, 0)), 1),
    (np.array([[1.0, np.inf], [2.0, 3.0]]), 1),
    (np.ones((3, 2)), 0),
    (np.ones((3, 2)), 3),
    (np.ones((3, 2)), True),
)
for _X_bad_fit_p23, _k_bad_fit_p23 in _invalid_fit_p23:
    _bad_eigh_calls_p23 = []
    def _unexpected_eigh_p23(matrix):
        _bad_eigh_calls_p23.append(np.array(matrix, copy=True))
        return _ORIGINAL_EIGH_P23(matrix)
    np.linalg.eigh = _unexpected_eigh_p23
    try:
        NumpyPCA(_k_bad_fit_p23).fit(_X_bad_fit_p23)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid fit input must raise ValueError")
    finally:
        np.linalg.eigh = _ORIGINAL_EIGH_P23
    assert _bad_eigh_calls_p23 == []

for _bad_X_transform_p23 in (np.ones(4), np.empty((0, 4)), np.ones((2, 5)), np.array([[1.0, 2.0, np.nan, 4.0]])):
    try:
        _model_p23.transform(_bad_X_transform_p23)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid transform input must raise ValueError")

for _bad_Z_p23 in (np.ones(2), np.empty((0, 2)), np.ones((3, 3)), np.array([[1.0, np.nan]])):
    try:
        _model_p23.inverse_transform(_bad_Z_p23)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid inverse-transform input must raise ValueError")
try:
    NumpyPCA(2).inverse_transform(np.ones((2, 2)))
except ValueError:
    pass
else:
    raise AssertionError("inverse_transform before fit must raise ValueError")